<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 09 — Regressão Linear
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Prevendo o Preço das Passagens do Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📈 Regressão</span>
</div>


## Uma virada importante no curso

Até agora todos os modelos que treinamos resolviam o **mesmo tipo de problema**:
classificar se um passageiro sobreviveu ou não — uma resposta binária (0 ou 1).

Hoje mudamos de problema. Em vez de prever **uma categoria**, vamos prever **um número**.

```
CLASSIFICAÇÃO (aulas anteriores)        REGRESSÃO (hoje)
─────────────────────────────────────   ──────────────────────────────────
Pergunta: "Esse passageiro sobreviveu?" Pergunta: "Quanto custou a passagem?"
Resposta: SIM ou NÃO                    Resposta: £ 7.25  ou  £ 156.50
Algoritmos: KNN, Log. Reg., SVM,        Algoritmo: Regressão Linear
            Árvore de Decisão
```

### O problema de hoje: quanto valia uma passagem do Titanic?

O preço das passagens variava enormemente — de algumas libras para a 3ª classe
até centenas de libras para suítes de 1ª classe. Vamos construir um modelo que,
dadas as características de um passageiro (classe, gênero, família a bordo),
consiga estimar o quanto ele pagou pela passagem.

Esse tipo de problema aparece em todo lugar na vida real:
estimar preço de imóveis, prever vendas, calcular seguros, definir salários.

---

## Roteiro de hoje

| Parte | Tema | 
|-------|------|
| **Config** | Preparando o problema de regressão no Titanic | 
| **1** | De classificação para regressão — o que muda? | 
| **2** | A reta dos mínimos quadrados — cálculo manual |
| **3** | O coeficiente R² — o que o modelo explica? | 
| **4** | Regressão Linear Múltipla no Titanic | 
| **5** | Resíduos — diagnosticando o modelo | 
| **6** | Métricas de regressão — MAE, MSE, RMSE e R² |

---

## Configuração — Execute antes de começar


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Carregando e preparando o Titanic ─────────────────────────────────────────
df_raw = sns.load_dataset("titanic")
df = df_raw.copy()

# Limpeza básica (mesma das aulas anteriores)
df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

# Features auxiliares
df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sozinho"]         = (df["tamanho_familia"] == 1).astype(int)
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# ── NOVO: definindo o ALVO de regressão ───────────────────────────────────────
# Vamos prever 'fare' (tarifa paga pela passagem)
# Removendo os poucos registros com fare = 0 (dados inconsistentes)
df = df[df["fare"] > 0].reset_index(drop=True)

print("✅ Dataset pronto para REGRESSÃO!")
print(f"   {len(df)} passageiros com tarifa > 0")
print()
print("Distribuição da variável ALVO — fare (£):")
print(df["fare"].describe().round(2).to_string())
print()
print(f"A passagem mais barata custou: £{df['fare'].min():.2f}")
print(f"A mais cara custou:            £{df['fare'].max():.2f}")
print(f"A mediana foi:                 £{df['fare'].median():.2f}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">De Classificação para Regressão — O que Muda?</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Mesmo raciocínio, novo tipo de resposta."</p></div></div></div>


### O que muda na prática?

| Aspecto | Classificação | Regressão |
|---------|--------------|-----------|
| **Variável Y** | Categoria (0, 1, A, B…) | Número contínuo (preço, idade, temperatura) |
| **Saída do modelo** | Classe + probabilidade | Valor numérico estimado |
| **Métricas** | Acurácia, F1, AUC-ROC | MAE, RMSE, R² |
| **Pergunta típica** | "Qual categoria?" | "Quanto?" ou "Quando?" |
| **Visualização** | Matriz de confusão, ROC | Gráfico de resíduos, previsto vs real |

### Exemplos reais de regressão

| Problema | Features (X) | Alvo (Y) |
|----------|-------------|----------|
| Preço de imóveis | Área, bairro, quartos | Preço em R$ |
| Consumo de combustível | Velocidade, carga, motor | Litros/100km |
| Previsão de vendas | Sazonalidade, promoção, histórico | Unidades vendidas |
| Risco de crédito | Renda, dívidas, histórico | Score numérico |
| **Titanic (hoje)** | **Classe, gênero, família** | **£ da passagem** |


In [ ]:
# Visualizando o nosso alvo: distribuição das tarifas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Variável Alvo — Fare (Tarifa da Passagem) £", fontweight="bold")

# Histograma
axes[0].hist(df["fare"], bins=40, color="#0f3460", edgecolor="white", alpha=0.85)
axes[0].axvline(df["fare"].mean(),   color="#e94560", linestyle="--",
                linewidth=2, label=f"Média: £{df['fare'].mean():.0f}")
axes[0].axvline(df["fare"].median(), color="#f0a500", linestyle="--",
                linewidth=2, label=f"Mediana: £{df['fare'].median():.0f}")
axes[0].set_xlabel("Tarifa (£)"); axes[0].set_ylabel("Frequência")
axes[0].set_title("Distribuição das Tarifas")
axes[0].legend()

# Boxplot por classe
df.boxplot(column="fare", by="pclass", ax=axes[1],
           boxprops=dict(color="#0f3460"),
           medianprops=dict(color="#e94560", linewidth=2.5),
           whiskerprops=dict(color="#0f3460"),
           capprops=dict(color="#0f3460"))
axes[1].set_xlabel("Classe"); axes[1].set_ylabel("Tarifa (£)")
axes[1].set_title("Tarifa por Classe")
plt.sca(axes[1]); plt.title("Tarifa por Classe"); plt.suptitle("")

# Scatter: tamanho da família vs tarifa
sc = axes[2].scatter(df["tamanho_familia"], df["fare"],
                      c=df["pclass"], cmap="RdYlBu_r",
                      s=25, alpha=0.5, edgecolors="none")
plt.colorbar(sc, ax=axes[2], label="Classe (1=rico, 3=pobre)")
axes[2].set_xlabel("Tamanho da Família")
axes[2].set_ylabel("Tarifa (£)")
axes[2].set_title("Família vs Tarifa (colorido por classe)")

plt.tight_layout()
plt.savefig("aula09_distribuicao_fare.png", dpi=110, bbox_inches="tight")
plt.show()

print("Observações importantes:")
print("  Distribuição muito assimétrica — maioria paga pouco, poucos pagam muito")
print("  Diferença enorme entre as classes")
print("  Famílias maiores tendem a pagar mais (mais passageiros = mais tickets)")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Antes de treinar qualquer modelo, analise os gráficos e responda: (a) por que a média é tão maior que a mediana na distribuição da tarifa? (b) a relação entre classe e tarifa é linear? (c) você esperaria um R² alto ou baixo para esse problema? Por quê?</span></div>

*✏️ (a) Média > Mediana porque: `???`*

*✏️ (b) Relação classe x tarifa é linear? `???`*

*✏️ (c) Espero R² `???` porque: `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A Reta dos Mínimos Quadrados — Cálculo Manual</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Antes de confiar na caixa-preta, entenda o que ela calcula."</p></div></div></div>


### A equação da reta

A Regressão Linear Simples aprende uma reta que relaciona X e Y:

```
ŷ = β₀ + β₁ · x

  β₀ = intercepto  (valor de ŷ quando x = 0)
  β₁ = inclinação  (quanto ŷ varia para cada +1 unidade de x)
  ŷ  = valor previsto ("y chapéu")
  ε  = resíduo = y − ŷ  (erro da previsão)
```

### O critério: Mínimos Quadrados Ordinários (OLS)

A "melhor reta" é aquela que **minimiza a soma dos erros ao quadrado**:

```
SQR = Σ (yᵢ − ŷᵢ)²   ← soma dos quadrados dos resíduos
```

Por que quadrado? Para penalizar erros grandes mais que erros pequenos,
e para que erros positivos e negativos não se cancelem.

### Fórmulas fechadas para β₁ e β₀

```
       Σ (xᵢ − x̄)(yᵢ − ȳ)        covariância(x, y)
β₁ = ─────────────────────── = ────────────────────
           Σ (xᵢ − x̄)²               variância(x)

β₀ = ȳ − β₁ · x̄
```

A reta **sempre passa pelo ponto (x̄, ȳ)** — o centro de gravidade dos dados.


In [ ]:
# Calculando β₀ e β₁ MANUALMENTE — exemplo simples antes do Titanic
# Usaremos: x = pclass (1, 2, 3) e y = fare médio por classe

import numpy as np
import pandas as pd

# Dados simplificados para entender o cálculo
x = np.array([1.0, 2.0, 3.0])   # classe
y = np.array([df[df.pclass==1]["fare"].mean(),
              df[df.pclass==2]["fare"].mean(),
              df[df.pclass==3]["fare"].mean()])

print("CÁLCULO MANUAL DA RETA — Classe vs Tarifa Média")
print("=" * 55)
print(f"Dados:")
print(f"  x (classe):          {x}")
print(f"  y (tarifa média £):  {y.round(2)}")

# Passo 1: médias
x_bar = x.mean()
y_bar = y.mean()
print(f"\nPasso 1 — Médias:")
print(f"  x̄ = {x_bar:.2f}    ȳ = £{y_bar:.2f}")

# Passo 2: desvios
dx = x - x_bar
dy = y - y_bar
print(f"\nPasso 2 — Desvios (xᵢ − x̄) e (yᵢ − ȳ):")
for i in range(len(x)):
    print(f"  Classe {x[i]:.0f}: dx={dx[i]:+.1f}   dy={dy[i]:+.2f}   "
          f"dx·dy={dx[i]*dy[i]:+.2f}   dx²={dx[i]**2:.1f}")

# Passo 3: β₁ e β₀
numerador   = (dx * dy).sum()
denominador = (dx**2).sum()
beta1 = numerador / denominador
beta0 = y_bar - beta1 * x_bar

print(f"\nPasso 3 — Coeficientes:")
print(f"  Numerador   Σ(xᵢ−x̄)(yᵢ−ȳ) = {numerador:.2f}")
print(f"  Denominador Σ(xᵢ−x̄)²      = {denominador:.2f}")
print(f"  β₁ = {numerador:.2f} / {denominador:.2f} = {beta1:.2f}")
print(f"  β₀ = {y_bar:.2f} − {beta1:.2f} × {x_bar:.2f} = {beta0:.2f}")
print(f"\n  RETA:  ŷ = {beta0:.2f} + ({beta1:.2f}) · x")
print(f"\nInterpretação:")
print(f"  Para cada aumento de 1 classe (de 1ª para 2ª, ou 2ª para 3ª),")
print(f"  a tarifa prevista muda em £{beta1:.2f}.")
print(f"  Sinal negativo: classes mais altas numericamente = passagens mais baratas.")


In [ ]:
# Visualizando a reta e os resíduos — exemplo de 3 pontos
y_hat = beta0 + beta1 * x
residuos = y - y_hat

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Regressão Linear — Classe vs Tarifa Média (3 pontos)",
             fontweight="bold")

x_line = np.linspace(0.5, 3.5, 100)
y_line = beta0 + beta1 * x_line

# Gráfico 1: reta com resíduos
ax1.plot(x_line, y_line, color="#0f3460", linewidth=2.5,
         label=f"ŷ = {beta0:.1f} + ({beta1:.1f})·x")
ax1.scatter(x, y, color="#e94560", s=150, zorder=5,
            label="Tarifa média real", edgecolors="white", linewidth=1.5)
ax1.scatter(x, y_hat, color="#0f3460", marker="x", s=100, zorder=6,
            label="Previsão (ŷ)")

# Resíduos
for xi, yi, yi_hat in zip(x, y, y_hat):
    ax1.vlines(xi, yi_hat, yi, color="#f0a500", linewidth=2, linestyle="--")
    ax1.annotate(f"ε = {yi-yi_hat:.1f}",
                 xy=(xi + 0.05, (yi + yi_hat)/2),
                 fontsize=10, color="#f0a500")

ax1.scatter(x_bar, y_bar, color="#2ecc71", s=200, zorder=7,
            marker="*", label=f"Centro (x̄={x_bar:.1f}, ȳ=£{y_bar:.1f})")
ax1.set_xlabel("Classe"); ax1.set_ylabel("Tarifa (£)")
ax1.set_title("Reta OLS com Resíduos (ε)")
ax1.legend(fontsize=9)

# Gráfico 2: gráfico de resíduos
ax2.axhline(0, color="#0f3460", linewidth=1.5, linestyle="--")
ax2.bar(["1ª Classe","2ª Classe","3ª Classe"], residuos,
        color=["#0f3460" if r > 0 else "#e94560" for r in residuos],
        edgecolor="white", width=0.4)
for i, r in enumerate(residuos):
    ax2.text(i, r + (2 if r > 0 else -5), f"£{r:.1f}",
             ha="center", fontweight="bold", fontsize=11)
ax2.set_ylabel("Resíduo ε = y − ŷ (£)")
ax2.set_title("Resíduos por Classe")

plt.tight_layout()
plt.savefig("aula09_reta_residuos.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"Soma dos resíduos: {residuos.sum():.10f} ≈ 0  (propriedade matemática do OLS!)")
print("A soma dos resíduos em OLS é SEMPRE zero — os erros se cancelam.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Calcule manualmente β₁ e β₀ para o mini-dataset abaixo usando as fórmulas vistas. Depois use numpy para verificar. Interprete o coeficiente β₁ no contexto do problema.</span></div>

### Mini-dataset para cálculo manual

| Passageiro | Tamanho da Família | Tarifa (£) |
|------------|-------------------|-----------|
| A | 1 | 7.25 |
| B | 2 | 15.50 |
| C | 3 | 24.00 |
| D | 4 | 31.80 |
| E | 6 | 52.00 |

*✏️ x̄ = `???`   ȳ = `???`*

*✏️ β₁ = `???`   β₀ = `???`*

*✏️ Reta: ŷ = `???` + `???` · x*

*✏️ Interpretação de β₁: a cada membro adicional na família, a tarifa muda em £`???`*


In [ ]:
# ✏️ Verifique seu cálculo manual aqui

x_m = np.array([1.0, 2.0, 3.0, 4.0, 6.0])
y_m = np.array([7.25, 15.50, 24.00, 31.80, 52.00])

# ✏️ Calcule manualmente x_bar, y_bar, beta1, beta0
# x_bar = ???
# y_bar = ???
# beta1 = ???
# beta0 = ???

# Verificação com numpy (polyfit — ajusta polinômio de grau 1 = reta)
beta1_np, beta0_np = np.polyfit(x_m, y_m, 1)
print(f"Verificação numpy:")
print(f"  β₁ = {beta1_np:.4f}")
print(f"  β₀ = {beta0_np:.4f}")
print(f"  Reta: ŷ = {beta0_np:.2f} + {beta1_np:.2f} · x")
print()
print("Compare com seu cálculo manual. Os valores devem ser idênticos!")


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# x_m = np.array([1.0, 2.0, 3.0, 4.0, 6.0])
# y_m = np.array([7.25, 15.50, 24.00, 31.80, 52.00])
#
# x_bar = x_m.mean()
# y_bar = y_m.mean()
#
# dx = x_m - x_bar
# dy = y_m - y_bar
#
# beta1 = (dx * dy).sum() / (dx**2).sum()
# beta0 = y_bar - beta1 * x_bar
#
# print("GABARITO — Cálculo Manual:")
# print(f"  x̄ = {x_bar:.2f}   ȳ = {y_bar:.2f}")
# print()
# print("  Tabela de desvios:")
# for xi, yi, dxi, dyi in zip(x_m, y_m, dx, dy):
#     print(f"  x={xi:.0f}  y={yi:.2f}  dx={dxi:+.2f}  dy={dyi:+.2f}  "
#           f"dx·dy={dxi*dyi:+.3f}  dx²={dxi**2:.2f}")
# print()
# print(f"  Σ(dx·dy) = {(dx*dy).sum():.4f}")
# print(f"  Σ(dx²)   = {(dx**2).sum():.4f}")
# print(f"  β₁ = {beta1:.4f}")
# print(f"  β₀ = {beta0:.4f}")
# print(f"  Reta: ŷ = {beta0:.2f} + {beta1:.2f} · x")
# print()
# print("Interpretação de β₁:")
# print(f"  Para cada membro adicional na família, a tarifa aumenta em £{beta1:.2f}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Coeficiente R² — O que o Modelo Explica?</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O R² responde: qual fração da variação de Y meu modelo captura?"</p></div></div></div>


### Definição

O **R² (coeficiente de determinação)** mede a proporção da variância de Y
que é explicada pelo modelo:

```
          Σ (yᵢ − ŷᵢ)²       Soma dos Quadrados dos Resíduos (SQR)
R² = 1 − ────────────────  =  1  −  ──────────────────────────────
          Σ (yᵢ − ȳ)²         Soma dos Quadrados Totais     (SQT)
```

| R² | Interpretação |
|----|---------------|
| **1.00** | Modelo perfeito — prevê Y sem nenhum erro |
| **0.75** | Modelo explica 75% da variação de Y |
| **0.00** | Modelo não explica nada (equivale a sempre prever ȳ) |
| **< 0** | Modelo é pior do que simplesmente usar a média! |

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#856404;">⚠️ </strong><span style="color:#856404;">Um R² alto não garante um bom modelo — pode ser overfitting. Um R² baixo não significa modelo inútil — depende do problema. Prever o preço exato de ações tem R² próximo de 0 mesmo com modelos sofisticados.</span></div>


In [ ]:
# Calculando R² manualmente — passo a passo
print("CÁLCULO MANUAL DO R² — Mini-dataset família vs tarifa")
print("=" * 58)

x_m = np.array([1.0, 2.0, 3.0, 4.0, 6.0])
y_m = np.array([7.25, 15.50, 24.00, 31.80, 52.00])

beta1_m, beta0_m = np.polyfit(x_m, y_m, 1)
y_hat_m = beta0_m + beta1_m * x_m
y_bar_m = y_m.mean()

# Tabela de cálculo
print(f"\n  {'xᵢ':>4} {'yᵢ':>7} {'ŷᵢ':>7} {'(yᵢ-ŷᵢ)':>10} {'(yᵢ-ŷᵢ)²':>12} {'(yᵢ-ȳ)²':>10}")
print("  " + "-"*55)
SQR = 0; SQT = 0
for xi, yi, yi_hat in zip(x_m, y_m, y_hat_m):
    res   = yi - yi_hat
    res2  = res**2
    dev2  = (yi - y_bar_m)**2
    SQR += res2; SQT += dev2
    print(f"  {xi:>4.0f} {yi:>7.2f} {yi_hat:>7.2f} "
          f"{res:>10.2f} {res2:>12.4f} {dev2:>10.4f}")

print("  " + "-"*55)
print(f"  {'SOMA':>4} {'':>7} {'':>7} {'':>10} {SQR:>12.4f} {SQT:>10.4f}")
print()
print(f"SQR = Σ(yᵢ−ŷᵢ)² = {SQR:.4f}")
print(f"SQT = Σ(yᵢ−ȳ)²  = {SQT:.4f}")
print(f"R² = 1 − SQR/SQT = 1 − {SQR:.4f}/{SQT:.4f} = {1-SQR/SQT:.4f}")
print()
print(f"O modelo explica {(1-SQR/SQT):.0%} da variação nas tarifas.")


In [ ]:
# Visualizando o que SQR e SQT representam geometricamente
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("R² — Visualização Geométrica", fontweight="bold")

x_plot = np.linspace(0.5, 6.5, 100)
y_plot = beta0_m + beta1_m * x_plot

# Gráfico 1: SQT — variação total (em torno da média)
ax1 = axes[0]
ax1.plot(x_plot, y_plot, color="#0f3460", linewidth=2, label="Reta OLS")
ax1.axhline(y_bar_m, color="#f0a500", linestyle="--",
            linewidth=2, label=f"Média ȳ = {y_bar_m:.1f}")
ax1.scatter(x_m, y_m, color="#e94560", s=100, zorder=5, label="Dados reais")
for xi, yi in zip(x_m, y_m):
    ax1.vlines(xi, y_bar_m, yi, color="#f0a500", linewidth=1.5, alpha=0.7)
ax1.set_title("SQT — Variação Total (distância de cada ponto até a média)",
              fontweight="bold")
ax1.set_xlabel("Tamanho da Família"); ax1.set_ylabel("Tarifa (£)")
ax1.legend(fontsize=9)

# Gráfico 2: SQR — variação residual (em torno da reta)
ax2 = axes[1]
ax2.plot(x_plot, y_plot, color="#0f3460", linewidth=2, label="Reta OLS")
ax2.scatter(x_m, y_m, color="#e94560", s=100, zorder=5, label="Dados reais")
ax2.scatter(x_m, y_hat_m, color="#0f3460", marker="x",
            s=80, zorder=6, label="Previsões (ŷ)")
for xi, yi, yi_hat in zip(x_m, y_m, y_hat_m):
    ax2.vlines(xi, yi_hat, yi, color="#e94560", linewidth=1.5, alpha=0.7)
ax2.set_title(f"SQR — Variação Residual  |  R² = {1-SQR/SQT:.4f} "
              "(distância de cada ponto até a reta)", fontweight="bold")
ax2.set_xlabel("Tamanho da Família"); ax2.set_ylabel("Tarifa (£)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("aula09_r2_geometria.png", dpi=110, bbox_inches="tight")
plt.show()

print("R² = 1 − SQR/SQT")
print("     1 → reta passa em todos os pontos (resíduos = 0)")
print("     0 → reta não melhora nada (tão boa quanto a média)")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Interprete o R² calculado: (a) o valor obtido indica um bom ou mau modelo? O que ele diz sobre a relação entre tamanho da família e tarifa? (b) se o R² fosse 0.0, o que isso significaria na prática? (c) é possível ter R² negativo? Quando isso aconteceria?</span></div>

*✏️ (a) R² = `???` indica: `???`*

*✏️ (b) R² = 0.0 significaria: `???`*

*✏️ (c) R² negativo: `???` — isso aconteceria quando: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# r2_val = 1 - SQR/SQT
# print("Gabarito:")
# print()
# print(f"(a) R² = {r2_val:.4f} = {r2_val:.0%}")
# print("    Indica um modelo MUITO BOM para este mini-dataset.")
# print("    O tamanho da família explica a maior parte da variação na tarifa.")
# print("    Isso faz sentido: mais pessoas na família = mais passagens compradas.")
# print()
# print("(b) R² = 0.0 significaria:")
# print("    O modelo não acrescenta nada.")
# print("    Prever sempre a média (ȳ) seria tão bom quanto usar a reta.")
# print("    A feature x não tem relação linear com y.")
# print()
# print("(c) R² negativo é possível!")
# print("    Acontece quando a reta é PIOR do que usar a média.")
# print("    Geralmente sinal de problema: feature errada, outliers extremos,")
# print("    ou uso de um modelo inadequado para o padrão dos dados.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Regressão Linear Múltipla no Titanic</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Uma feature raramente é suficiente — combinamos várias para prever melhor."</p></div></div></div>


### De simples para múltipla

A **Regressão Linear Simples** usa uma única feature:
```
ŷ = β₀ + β₁ · x₁
```

A **Regressão Linear Múltipla** combina várias features:
```
ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₙ·xₙ
```

Cada βᵢ representa o impacto de xᵢ na previsão, **mantendo todas as outras features constantes**.
Isso é chamado de **efeito parcial** — isolamos a contribuição de cada variável.

### Pipeline para o Titanic

Vamos prever `fare` usando as features que preparamos ao longo do curso.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ── Preparando features para REGRESSÃO ────────────────────────────────────────
# Alvo: fare (tarifa)
# Removemos fare e variáveis derivadas de fare das features
embarked_ohe = pd.get_dummies(df["embarked"], prefix="emb", drop_first=True)
titulo_temp  = df["name"].str.extract(r",\s([A-Za-z]+)\.").iloc[:,0]                  .map(lambda t: t if t in ["Mr","Miss","Mrs","Master"] else "Raro")
titulo_ohe   = pd.get_dummies(titulo_temp, prefix="titulo", drop_first=True)

FEATURES_REG = ["pclass","sex_enc","age","tamanho_familia","sozinho"]
df_reg = pd.concat([df[FEATURES_REG], embarked_ohe, titulo_ohe], axis=1)
y_reg  = df["fare"].copy()

# Garantindo sem NaN
df_reg = df_reg.fillna(0)

X_tr, X_te, y_tr, y_te = train_test_split(
    df_reg, y_reg, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_tr_sc    = scaler_reg.fit_transform(X_tr)
X_te_sc    = scaler_reg.transform(X_te)

# Treinando o modelo
lr = LinearRegression()
lr.fit(X_tr_sc, y_tr)

y_pred_tr = lr.predict(X_tr_sc)
y_pred_te = lr.predict(X_te_sc)

print("✅ Regressão Linear Múltipla treinada!")
print()
print(f"  Intercepto (β₀):  £{lr.intercept_:.2f}")
print(f"  R² no treino:     {r2_score(y_tr, y_pred_tr):.4f}")
print(f"  R² no teste:      {r2_score(y_te, y_pred_te):.4f}")
print(f"  MAE no teste:     £{mean_absolute_error(y_te, y_pred_te):.2f}")
print(f"  RMSE no teste:    £{np.sqrt(mean_squared_error(y_te, y_pred_te)):.2f}")


In [ ]:
# Visualizando os coeficientes — qual feature mais influencia a tarifa?
colunas_reg = df_reg.columns.tolist()
coef_df = pd.DataFrame({
    "feature":    colunas_reg,
    "coeficiente": lr.coef_
}).sort_values("coeficiente", key=abs, ascending=False)

print("Coeficientes da Regressão Linear Múltipla:")
print("(dados normalizados — coeficientes são comparáveis)")
print("=" * 52)
for _, row in coef_df.iterrows():
    sinal = "⬆" if row.coeficiente > 0 else "⬇"
    print(f"  {row.feature:<20} {row.coeficiente:>+10.4f}  {sinal}")


In [ ]:
# Gráfico de coeficientes e previsões
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Regressão Linear Múltipla — Tarifa do Titanic",
             fontweight="bold")

# Gráfico 1: coeficientes
import matplotlib.patches as mpatches
cores_coef = ["#0f3460" if v > 0 else "#e94560"
              for v in coef_df["coeficiente"]]
axes[0].barh(coef_df["feature"], coef_df["coeficiente"],
             color=cores_coef, edgecolor="white", height=0.65)
axes[0].axvline(0, color="black", linewidth=1.2)
for i, (_, row) in enumerate(coef_df.iterrows()):
    x_txt = row.coeficiente + (0.3 if row.coeficiente >= 0 else -0.3)
    ha = "left" if row.coeficiente >= 0 else "right"
    axes[0].text(x_txt, i, f"{row.coeficiente:+.2f}",
                 va="center", ha=ha, fontsize=9)
axes[0].set_xlabel("Coeficiente β (normalizado)")
axes[0].set_title("Impacto de cada Feature na Tarifa "
                  "Azul = aumenta | Vermelho = diminui",
                  fontweight="bold")
axes[0].invert_yaxis()

# Gráfico 2: previsto vs real
max_fare = min(y_te.max(), 300)
axes[1].scatter(y_te, y_pred_te, alpha=0.4, color="#0f3460",
                s=20, edgecolors="none")
axes[1].plot([0, max_fare], [0, max_fare], color="#e94560",
             linestyle="--", linewidth=2, label="Previsão perfeita")
axes[1].set_xlabel("Tarifa Real (£)")
axes[1].set_ylabel("Tarifa Prevista (£)")
axes[1].set_title(f"Previsto vs Real
R² = {r2_score(y_te, y_pred_te):.3f}",
                  fontweight="bold")
axes[1].set_xlim(0, max_fare); axes[1].set_ylim(0, max_fare)
axes[1].legend()

plt.tight_layout()
plt.savefig("aula09_reg_multipla.png", dpi=110, bbox_inches="tight")
plt.show()

print("No gráfico Previsto vs Real:")
print("  Pontos na linha vermelha = previsão perfeita")
print("  Pontos acima = modelo superestimou")
print("  Pontos abaixo = modelo subestimou")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Analise os coeficientes e responda: (a) qual feature tem maior impacto POSITIVO na tarifa? Faz sentido? (b) qual feature tem maior impacto NEGATIVO? Por quê? (c) olhando o gráfico Previsto vs Real, para quais valores de tarifa o modelo erra mais? Isso é um problema com os dados ou com o modelo?</span></div>

*✏️ (a) Maior impacto positivo: `???` — faz sentido porque: `???`*

*✏️ (b) Maior impacto negativo: `???` — porque: `???`*

*✏️ (c) O modelo erra mais para tarifas `???` — isso acontece porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# top_pos = coef_df[coef_df.coeficiente > 0].iloc[0]
# top_neg = coef_df[coef_df.coeficiente < 0].iloc[0]
# print("Gabarito:")
# print()
# print(f"(a) Maior impacto POSITIVO: {top_pos.feature} (β={top_pos.coeficiente:+.3f})")
# print("    Faz sentido: pclass é o maior determinante do preço da passagem.")
# print("    1ª classe é significativamente mais cara que 2ª e 3ª.")
# print()
# print(f"(b) Maior impacto NEGATIVO: {top_neg.feature} (β={top_neg.coeficiente:+.3f})")
# print("    Na codificação usada, pclass=3 tem valor maior numericamente.")
# print("    Ou: ser Mr/homem adulto está associado a passagens mais baratas")
# print("    (homens eram mais comuns na 3ª classe).")
# print()
# print("(c) O modelo erra mais para tarifas ALTAS (>£100)")
# print("    Razão: outliers — poucas passagens caríssimas, difíceis de prever.")
# print("    A distribuição é muito assimétrica (cauda longa à direita).")
# print("    Solução possível: transformar Y com log antes de modelar.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Resíduos — Diagnosticando o Modelo</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Os erros do modelo contam uma história — aprenda a lê-la."</p></div></div></div>


### Por que analisar os resíduos?

Os **resíduos** são os erros individuais do modelo: `εᵢ = yᵢ − ŷᵢ`

Para que a Regressão Linear seja válida, os resíduos devem obedecer 4 pressupostos:

| Pressuposto | O que significa | Como verificar |
|-------------|----------------|----------------|
| **Linearidade** | Relação Y\|X é linear | Resíduo vs Previsto — sem padrão curvo |
| **Homocedasticidade** | Variância dos resíduos é constante | Resíduo vs Previsto — sem funil |
| **Normalidade** | Resíduos têm distribuição normal | Histograma + Q-Q plot |
| **Independência** | Resíduos não são correlacionados | Resíduo vs Ordem — sem tendência |

Se um pressuposto for violado, as previsões ainda funcionam mas os **intervalos de confiança
e testes estatísticos** ficam comprometidos.


In [ ]:
# Calculando e analisando os resíduos
residuos_te = y_te.values - y_pred_te

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Diagnóstico de Resíduos — Regressão Linear no Titanic",
             fontsize=13, fontweight="bold")

# Gráfico 1: Resíduo vs Previsto (pressuposto de linearidade e homocedasticidade)
axes[0,0].scatter(y_pred_te, residuos_te, alpha=0.4, color="#0f3460",
                  s=20, edgecolors="none")
axes[0,0].axhline(0, color="#e94560", linestyle="--", linewidth=2)
axes[0,0].set_xlabel("Valor Previsto (ŷ)")
axes[0,0].set_ylabel("Resíduo (ε = y − ŷ)")
axes[0,0].set_title("Resíduo vs Previsto "
                    "Ideal: pontos aleatórios em torno de zero, sem padrão",
                    fontweight="bold")

# Anotando o padrão problemático
axes[0,0].annotate("Padrão em funil? → Heterocedasticidade",
                   xy=(150, 200), fontsize=9, color="#e94560",
                   bbox=dict(boxstyle="round", facecolor="#fff3cd", alpha=0.8))

# Gráfico 2: Histograma dos resíduos (pressuposto de normalidade)
axes[0,1].hist(residuos_te, bins=35, color="#0f3460",
               edgecolor="white", alpha=0.85, density=True)
# Curva normal teórica
mu, sigma = residuos_te.mean(), residuos_te.std()
x_norm = np.linspace(residuos_te.min(), residuos_te.max(), 200)
y_norm = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x_norm-mu)/sigma)**2)
axes[0,1].plot(x_norm, y_norm, color="#e94560", linewidth=2,
               label="Normal teórica")
axes[0,1].axvline(0, color="#f0a500", linestyle="--", linewidth=1.5)
axes[0,1].set_xlabel("Resíduo (£)")
axes[0,1].set_ylabel("Densidade")
axes[0,1].set_title("Distribuição dos Resíduos Ideal: curva em sino centrada em zero",
                    fontweight="bold")
axes[0,1].legend()

# Gráfico 3: Q-Q plot (normalidade mais preciso)
from scipy import stats
(qq_x, qq_y), (slope, intercept, r) = stats.probplot(residuos_te,
                                                        dist="norm",
                                                        fit=True)[:2],                                         stats.probplot(residuos_te, dist="norm")[1]
axes[1,0].scatter(qq_x, qq_y, alpha=0.5, color="#0f3460", s=20)
x_line = np.array([qq_x.min(), qq_x.max()])
axes[1,0].plot(x_line, slope*x_line + intercept, color="#e94560",
               linewidth=2, label="Linha normal")
axes[1,0].set_xlabel("Quantis Teóricos (Normal)")
axes[1,0].set_ylabel("Quantis dos Resíduos")
axes[1,0].set_title("Q-Q Plot Ideal: pontos na linha vermelha", fontweight="bold")
axes[1,0].legend()

# Gráfico 4: Resíduo vs Ordem (independência)
axes[1,1].plot(range(len(residuos_te)), residuos_te,
               color="#0f3460", linewidth=0.8, alpha=0.6)
axes[1,1].axhline(0, color="#e94560", linestyle="--", linewidth=2)
axes[1,1].fill_between(range(len(residuos_te)), residuos_te,
                        alpha=0.15, color="#0f3460")
axes[1,1].set_xlabel("Ordem da Observação")
axes[1,1].set_ylabel("Resíduo (£)")
axes[1,1].set_title("Resíduo vs Ordem Ideal: sem tendência ou padrão temporal",
                    fontweight="bold")

plt.tight_layout()
plt.savefig("aula09_residuos.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# Diagnosticando os pressupostos quantitativamente
print("DIAGNÓSTICO DOS PRESSUPOSTOS — Regressão Linear")
print("=" * 55)

# 1. Linearidade: correlação entre resíduo e previsto (deve ser ~0)
corr_res_pred = np.corrcoef(y_pred_te, residuos_te)[0,1]
print(f"\n1. Linearidade:")
print(f"   Correlação (resíduo × previsto): {corr_res_pred:.4f}")
print(f"   Avaliação: {'✅ OK (próximo de zero)' if abs(corr_res_pred) < 0.1 else '⚠️ Possível não-linearidade'}")

# 2. Homocedasticidade: variância dos resíduos por faixa de previsto
tercis = np.percentile(y_pred_te, [33, 66])
mask1 = y_pred_te < tercis[0]
mask2 = (y_pred_te >= tercis[0]) & (y_pred_te < tercis[1])
mask3 = y_pred_te >= tercis[1]
std1, std2, std3 = (residuos_te[mask1].std(),
                    residuos_te[mask2].std(),
                    residuos_te[mask3].std())
print(f"\n2. Homocedasticidade (variância dos resíduos por faixa de ŷ):")
print(f"   Baixo ŷ  (< £{tercis[0]:.0f}): std = £{std1:.2f}")
print(f"   Médio ŷ  (£{tercis[0]:.0f}–£{tercis[1]:.0f}):  std = £{std2:.2f}")
print(f"   Alto ŷ   (> £{tercis[1]:.0f}): std = £{std3:.2f}")
razao = std3 / std1
print(f"   Razão max/min = {razao:.1f}x")
print(f"   Avaliação: {'⚠️ Heterocedasticidade detectada!' if razao > 3 else '✅ OK'}")

# 3. Normalidade: teste de Shapiro-Wilk (amostra dos resíduos)
from scipy.stats import shapiro
amostra_res = residuos_te[np.random.choice(len(residuos_te), 50, replace=False)]
stat, p_valor = shapiro(amostra_res)
print(f"\n3. Normalidade dos resíduos (Shapiro-Wilk, n=50):")
print(f"   Estatística: {stat:.4f} | p-valor: {p_valor:.4f}")
print(f"   Avaliação: {'✅ Normal (p > 0.05)' if p_valor > 0.05 else '⚠️ Não-normal (p ≤ 0.05)'}")

print(f"\nConclusão:")
print(f"  O principal problema é a HETEROCEDASTICIDADE — a variância dos erros")
print(f"  aumenta para tarifas altas. Solução: transformar Y com log(Y).")


In [ ]:
# Solução: transformação logarítmica do alvo
print("SOLUÇÃO — Transformação log(fare)")
print("=" * 50)

y_log_tr = np.log1p(y_tr)   # log(1 + y) — evita log(0)
y_log_te = np.log1p(y_te)

lr_log = LinearRegression()
lr_log.fit(X_tr_sc, y_log_tr)

y_pred_log_tr = lr_log.predict(X_tr_sc)
y_pred_log_te = lr_log.predict(X_te_sc)

# Convertendo de volta para £
y_pred_real_tr = np.expm1(y_pred_log_tr)   # inverso de log1p
y_pred_real_te = np.expm1(y_pred_log_te)

r2_orig = r2_score(y_te, y_pred_te)
r2_log  = r2_score(y_te, y_pred_real_te)
mae_orig = mean_absolute_error(y_te, y_pred_te)
mae_log  = mean_absolute_error(y_te, y_pred_real_te)

print(f"  {'Métrica':<12} {'Sem log':>12} {'Com log(y)':>14}  {'Melhorou?'}")
print("  " + "-"*50)
print(f"  {'R²':<12} {r2_orig:>12.4f} {r2_log:>14.4f}  "
      f"{'✅' if r2_log > r2_orig else '❌'}")
print(f"  {'MAE (£)':<12} {mae_orig:>12.2f} {mae_log:>14.2f}  "
      f"{'✅' if mae_log < mae_orig else '❌'}")

# Comparando resíduos antes e depois
res_log = y_te.values - y_pred_real_te
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Efeito da Transformação log(Y) nos Resíduos", fontweight="bold")

for ax, res, titulo, cor in [
    (axes[0], residuos_te,   "Sem transformação (Original)", "#e94560"),
    (axes[1], res_log,       "Com log(Y)",                   "#0f3460"),
]:
    ax.scatter(y_pred_te, res, alpha=0.35, color=cor, s=15, edgecolors="none")
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Valor Previsto (£)")
    ax.set_ylabel("Resíduo (£)")
    ax.set_title(titulo, fontweight="bold")

plt.tight_layout()
plt.savefig("aula09_log_transform.png", dpi=110, bbox_inches="tight")
plt.show()


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 6</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Métricas de Regressão — MAE, MSE, RMSE e R²</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Métricas diferentes respondem perguntas diferentes sobre os erros."</p></div></div></div>


### As quatro métricas essenciais

| Métrica | Fórmula | Unidade | Interpretação |
|---------|---------|---------|---------------|
| **MAE** | Σ\|yᵢ−ŷᵢ\| / n | Mesma de Y | Erro médio absoluto — intuitivo |
| **MSE** | Σ(yᵢ−ŷᵢ)² / n | Y² | Penaliza erros grandes |
| **RMSE** | √MSE | Mesma de Y | Raiz do MSE — mesma unidade, mais intuitivo |
| **R²** | 1 − SQR/SQT | Adimensional (0 a 1) | Fração da variância explicada |

### Quando usar cada uma?

- **MAE**: quando todos os erros importam igualmente (robusto a outliers)
- **RMSE**: quando erros grandes são especialmente problemáticos
- **R²**: para comparar modelos em problemas diferentes (não depende da escala)

<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 </strong><span style="color:#5b2c8d;">Se o RMSE for muito maior que o MAE, isso indica a presença de <strong>outliers</strong> — poucos exemplos com erros muito grandes. A razão RMSE/MAE > 2 é um sinal de alerta.</span></div>


In [ ]:
# Calculando todas as métricas manualmente — passo a passo
print("CÁLCULO MANUAL DAS MÉTRICAS — Regressão Linear")
print("=" * 55)

n  = len(y_te)
y_real = y_te.values
y_prev = y_pred_te

erros_abs = np.abs(y_real - y_prev)
erros_sq  = (y_real - y_prev)**2

# MAE
mae  = erros_abs.mean()

# MSE
mse  = erros_sq.mean()

# RMSE
rmse = np.sqrt(mse)

# R²
sqr  = erros_sq.sum()
sqt  = ((y_real - y_real.mean())**2).sum()
r2   = 1 - sqr/sqt

print(f"\nMétricas calculadas manualmente (n={n}):")
print(f"  MAE  = Σ|yᵢ−ŷᵢ| / n     = {erros_abs.sum():.2f} / {n} = {mae:.2f} £")
print(f"  MSE  = Σ(yᵢ−ŷᵢ)² / n    = {erros_sq.sum():.2f} / {n} = {mse:.2f} £²")
print(f"  RMSE = √MSE               = √{mse:.2f} = {rmse:.2f} £")
print(f"  R²   = 1 − SQR/SQT        = 1 − {sqr:.2f}/{sqt:.2f} = {r2:.4f}")
print()
print(f"  Razão RMSE/MAE: {rmse/mae:.2f}x")
print(f"  {'⚠️  Razão alta — outliers afetando RMSE' if rmse/mae > 2 else '✅ Razão normal'}")

# Verificação com sklearn
print()
print("Verificação com scikit-learn:")
print(f"  MAE:  {mean_absolute_error(y_te, y_pred_te):.2f}  ✓")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_te, y_pred_te)):.2f}  ✓")
print(f"  R²:   {r2_score(y_te, y_pred_te):.4f}  ✓")


In [ ]:
# Comparação visual de todos os modelos testados nesta aula
resultados_finais = {
    "Simples (pclass)":           None,
    "Simples (família)":          None,
    "Múltipla (sem log)":         None,
    "Múltipla (com log)":         None,
}

# Calculando cada um
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Simples - pclass
lr_s1 = LinearRegression()
X_s1_tr = scaler_reg.fit_transform(X_tr[["pclass"]])
X_s1_te = scaler_reg.transform(X_te[["pclass"]])
lr_s1.fit(X_s1_tr, y_tr)
r2_s1  = r2_score(y_te, lr_s1.predict(X_s1_te))
mae_s1 = mean_absolute_error(y_te, lr_s1.predict(X_s1_te))

# 2. Simples - família
lr_s2 = LinearRegression()
X_s2_tr = scaler_reg.fit_transform(X_tr[["tamanho_familia"]])
X_s2_te = scaler_reg.transform(X_te[["tamanho_familia"]])
lr_s2.fit(X_s2_tr, y_tr)
r2_s2  = r2_score(y_te, lr_s2.predict(X_s2_te))
mae_s2 = mean_absolute_error(y_te, lr_s2.predict(X_s2_te))

# 3. Múltipla sem log
r2_m1  = r2_score(y_te, y_pred_te)
mae_m1 = mean_absolute_error(y_te, y_pred_te)

# 4. Múltipla com log
r2_m2  = r2_score(y_te, y_pred_real_te)
mae_m2 = mean_absolute_error(y_te, y_pred_real_te)

comparacao = pd.DataFrame({
    "Modelo":   ["Simples (pclass)", "Simples (família)",
                 "Múltipla (sem log)", "Múltipla (com log)"],
    "R²":       [r2_s1, r2_s2, r2_m1, r2_m2],
    "MAE (£)":  [mae_s1, mae_s2, mae_m1, mae_m2],
    "Features": [1, 1, len(df_reg.columns), len(df_reg.columns)],
})

print("COMPARAÇÃO FINAL — Todos os Modelos de Regressão")
print("=" * 60)
print(comparacao.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Evolução dos Modelos de Regressão — Titanic", fontweight="bold")

cores_4 = ["#a8d8ea","#0f3460","#f0a500","#2ecc71"]
nomes   = comparacao["Modelo"].tolist()

for ax, metrica, titulo_y, melhor in [
    (axes[0], "R²",       "R² (maior = melhor)", "max"),
    (axes[1], "MAE (£)", "MAE em £ (menor = melhor)", "min"),
]:
    barras = ax.bar(range(len(nomes)), comparacao[metrica],
                    color=cores_4, edgecolor="white", width=0.55)
    for b, v in zip(barras, comparacao[metrica]):
        ax.text(b.get_x()+b.get_width()/2, v + (0.005 if metrica=="R²" else 0.5),
                f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_xticks(range(len(nomes)))
    ax.set_xticklabels(nomes, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel(titulo_y)
    ax.set_title(titulo_y.split("(")[0].strip(), fontweight="bold")

plt.tight_layout()
plt.savefig("aula09_comparacao_final.png", dpi=110, bbox_inches="tight")
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 — Analise a tabela de comparação e responda: (a) qual modelo tem o melhor R²? Adicionar features sempre melhora o R²? (b) qual modelo tem o menor MAE? Isso contradiz o R²? (c) a transformação log(Y) valeu a pena? Como você justificaria essa decisão para alguém que não conhece estatística?</span></div>

*✏️ (a) Melhor R²: `???` — adicionar features `???` melhora o R² porque: `???`*

*✏️ (b) Menor MAE: `???` — contradiz o R²? `???`*

*✏️ (c) A transformação log valeu? `???` — justificativa simples: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) Melhor R²: Múltipla (com log) — provavelmente")
# print("    Adicionar features QUASE SEMPRE aumenta R² no treino.")
# print("    Mas pode diminuir R² no teste se houver overfitting.")
# print("    Por isso usamos TREINO E TESTE separados — o R² no teste é o real.")
# print()
# print("(b) Menor MAE:")
# print("    MAE e R² medem coisas diferentes.")
# print("    R² mede variância relativa — normalizado entre 0 e 1.")
# print("    MAE mede erro absoluto em £ — depende da escala.")
# print("    Um modelo pode ter R² menor mas MAE menor se errar mais uniformemente.")
# print()
# print("(c) Transformação log:")
# print("    Valeu quando reduz heterocedasticidade e melhora resíduos.")
# print("    Justificativa simples: 'Passagens caras têm variação proporcional,")
# print("    não absoluta. Log transforma variação proporcional em aditiva.'")
# print()
# print("    Exemplo: erro de £10 numa passagem de £12 é enorme.")
# print("             erro de £10 numa passagem de £300 é irrelevante.")
# print("    Log trata esses casos de forma adequada.")


---

## Regressão vs Classificação — Resumo Comparativo

| Aspecto | Classificação | Regressão Linear |
|---------|--------------|-----------------|
| **Tipo de Y** | Categoria | Número contínuo |
| **Algoritmos vistos** | KNN, Log. Reg., SVM, Árvore | Regressão Linear |
| **Métricas** | Acurácia, F1, AUC | MAE, RMSE, R² |
| **Visualização do erro** | Matriz de confusão | Previsto vs Real, Resíduos |
| **Pressuposto crítico** | — | Linearidade, homocedasticidade |
| **Transformação comum** | Encoding de Y | log(Y) para assimetria |

---

## Checklist — O que você sabe fazer agora

| Habilidade | Praticada hoje? |
|------------|----------------|
| Distinguir problemas de regressão de classificação | ☐ |
| Calcular β₀ e β₁ manualmente pelo método dos mínimos quadrados | ☐ |
| Calcular e interpretar o R² passo a passo | ☐ |
| Treinar Regressão Linear Simples e Múltipla com scikit-learn | ☐ |
| Interpretar coeficientes de uma regressão múltipla | ☐ |
| Criar e interpretar os 4 gráficos de diagnóstico de resíduos | ☐ |
| Identificar heterocedasticidade e aplicar transformação log | ☐ |
| Calcular MAE, MSE, RMSE e R² manualmente e com sklearn | ☐ |

---


## Reflexão final

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">Escreva em suas próprias palavras: (1) qual é a diferença fundamental entre classificação e regressão? (2) por que a soma dos resíduos em OLS é sempre zero? (3) quando você usaria MAE em vez de RMSE para avaliar um modelo?</span></div>


**✏️ Minha reflexão:**

1. Classificação vs Regressão: *...*

2. Soma dos resíduos = 0 porque: *...*

3. Usaria MAE em vez de RMSE quando: *...*


---

## O que vem a seguir?

```
Classificação:  KNN ✅  Log. Reg. ✅  SVM ✅  Árvore ✅
Regressão:      Regressão Linear ✅
Próximos:       Regularização (Ridge, Lasso)  →  Random Forest
```

Na próxima aula veremos como lidar com o overfitting em Regressão Linear
usando **regularização (Ridge e Lasso)** — uma técnica que penaliza
coeficientes excessivamente grandes.

---

## Referências

- Scikit-Learn LinearRegression: https://scikit-learn.org/stable/modules/linear_model.html
- Gauss, C. F. (1809). *Theoria Motus Corporum Coelestium* — origem do método OLS.
- James, G. et al. (2021). *An Introduction to Statistical Learning* (ISLR). Springer.
  PDF gratuito: https://www.statlearning.com/
- Géron, A. (2019). *Hands-on Machine Learning*, Cap. 4. O'Reilly.
